In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.chat_message_histories import RedisChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_community.document_loaders import PyPDFLoader

In [3]:
loader = PyPDFLoader("./[삼성전기]사업보고서(2026.03.10).pdf")

In [5]:
loader_data = loader.load()

In [6]:
len(loader_data)

352

In [8]:
loader_data[0:2]

[Document(metadata={'producer': 'iText® 5.4.0 ©2000-2012 1T3XT BVBA (AGPL-version)', 'creator': 'PyPDF', 'creationdate': '2026-03-10T17:50:56+09:00', 'moddate': '2026-03-10T17:50:56+09:00', 'source': './[삼성전기]사업보고서(2026.03.10).pdf', 'total_pages': 352, 'page': 0, 'page_label': 'i'}, page_content='목                 차\n사 업 보 고 서 ............................................................................................................................................1\n【 대표이사 등의 확인 】...........................................................................................................................2\nI. 회사의 개요 ............................................................................................................................................3\n1. 회사의 개요......................................................................................................................................3\n2. 회사의 연혁................................................................................

In [12]:
import requests
from langchain_community.document_loaders import WebBaseLoader
from bs4 import SoupStrainer

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [30]:
client_id = "YXw4dmid0O2qTQNfhcY2"
client_secret = "BksteLLT_U"
url = "https://openapi.naver.com/v1/search/news.json"
headers = {
    "X-Naver-Client-Id": client_id,
    "X-Naver-Client-Secret": client_secret
}
params = {'query' : '삼성전기', 'display' : 100, 'start': 1, "sort" : "sim"}
response = requests.get(url, headers=headers, params=params)

In [31]:
total_naver_url = [url['link'] for url in response.json()['items'] if 'naver.com' in url['link']]

In [32]:
total_naver_url

['https://n.news.naver.com/mnews/article/629/0000482727?sid=101',
 'https://n.news.naver.com/mnews/article/277/0005736371?sid=101',
 'https://n.news.naver.com/mnews/article/003/0013829232?sid=101',
 'https://n.news.naver.com/mnews/article/008/0005331583?sid=101',
 'https://n.news.naver.com/mnews/article/082/0001371528?sid=101',
 'https://n.news.naver.com/mnews/article/421/0008833041?sid=101',
 'https://n.news.naver.com/mnews/article/014/0005493655?sid=101',
 'https://n.news.naver.com/mnews/article/366/0001149515?sid=105',
 'https://n.news.naver.com/mnews/article/421/0008832921?sid=101',
 'https://n.news.naver.com/mnews/article/018/0006237743?sid=101',
 'https://n.news.naver.com/mnews/article/014/0005493414?sid=101',
 'https://n.news.naver.com/mnews/article/018/0006237685?sid=101',
 'https://n.news.naver.com/mnews/article/119/0003070254?sid=101',
 'https://n.news.naver.com/mnews/article/031/0001014409?sid=101',
 'https://n.news.naver.com/mnews/article/030/0003408620?sid=105',
 'https://

In [33]:
#id newsct_article
bs4_kwargs = {
    'parse_only' : SoupStrainer("div", id="newsct_article")
}
rt = WebBaseLoader(total_naver_url[0],             bs_kwargs=bs4_kwargs)

In [34]:
삼성전기뉴스 = [WebBaseLoader(x,bs_kwargs=bs4_kwargs).load()[0] for x in total_naver_url]

In [49]:
from tqdm import tqdm
삼성전기뉴스 = []
for x in tqdm(total_naver_url):
    삼성전기뉴스.append(WebBaseLoader(x,bs_kwargs=bs4_kwargs).load()[0] )

100%|██████████| 36/36 [00:05<00:00,  6.81it/s]


In [50]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [46]:
vectorstore = Chroma.from_documents(
    documents=삼성전기뉴스,
    embedding=embeddings,
    collection_name="samsungem",
    persist_directory="./naver_news_vecdb"
)

In [51]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

In [52]:
retriever.invoke("지분 구조에 대해서")

[Document(metadata={'source': 'https://n.news.naver.com/mnews/article/011/0004600677?sid=101'}, page_content='\n\n정기주주총회서 추가 투자 계획 밝혀\n\n\n\n장덕현 삼성전기 사장.장덕현 삼성전기(009150) 사장은 18일 차세대 반도체 기판인 FC-BGA와 관련해 “현재 생산능력보다 고객 요구가 50% 이상 많아 공장 확대를 계획하고 있다”고 밝혔다.장 사장은 이날 서울 서초구에서 열린 53기 정기주주총회 직후 기자들과 만나 이 같이 말했다.장 사장은 “인공지능(AI) 대규모 투자 확대와 로보택시 도입 가속화, 휴머노이드의 현장 배치 본격화 등 전자부품 채용 확대 기회를 적극 활용해 시장 성장률을 상회하는 매출 확대를 추진하겠다”고 강조했다. 장 사장은 휴머노이드 사업에 대해서는 ““일부 부품은 올해 하반기부터 양산이 시작될 것”이라며 “산업용에 적용이 점차 확대될 것”이라고 밝혔다.이날 주총에서는 재무제표 승인과 정관 일부 변경, 사외이사 선임, 이사 보수 한도의 승인 등 7개 안건이 모두 원안대로 가결됐다. 최종구 사외이사는 재선임됐으며 김미영·이종훈 사외이사가 신규 선임됐다. 이사회 의장에는 최종구 사외이사가 선임됐다.\n\n'),
 Document(metadata={'source': 'https://n.news.naver.com/mnews/article/011/0004600785?sid=101'}, page_content='\n\n■정기 주총 개최삼성전기 사업 구조, 로보택시 도입 등글로벌 트렌드 겨냥 피지컬 AI에 힘줘삼성SDI도 전방산업 호황에 ‘핑크빛’로봇 등 무게 실어 수주 다양화 포부\n\n\n\n장덕현 삼성전기 대표이사 사장이 18일 서울 서초구 엘타워에서 열린 제53회 정기 주주총회에서 2025년 경영 성과와 2026년 경영 계획을 발표하고 있다. 사진제공=삼성SDI삼성전기(009150)가 사업구조를 인공지능(AI) 서버와 전장, 휴머노이드 로봇